# Car ownership: how many cars a household owns

## What this model answers

How many cars does a household own: none, one, two, or three or more? The model uses household size, workers, income, age, dwelling, density and the accessibility a car adds. Each number of cars has its own coefficients against owning no car.

| | |
|---|---|
| Choice | Number of cars |
| Alternatives | 0, 1, 2, 3 or more cars (all available; no car is the reference with utility 0) |
| Decision makers | Maryland households (not group quarters) |
| Data | ACS PUMS 2014-2018 (5-year), Maryland |
| Estimation sample | 91,348 households |

## How to run

- Kernel: *Python (biogeme)* (`~/.venvs/biogeme`, Biogeme 3.3); the data rebuild also needs `python3` with geopandas. Then *Run All*.
- `RUN_DATA_PREP`: rebuild the estimation tables in a scratch folder and compare them with the shipped ones.
- `RUN_CHECK`: log-likelihood at the article coefficients, no estimation (about 0.1 min).
- `RUN_ESTIMATION`: run the Biogeme estimation (off by default; the results in `output/` are read instead).
- `RUN_HOLDOUT`: log-likelihood of the hold-out records.
- Figures of the hold-out check are read from `output/holdout_*` (copied there by the notebook builder).

## Contents

Article table: `est_cars`. This notebook goes from the raw records to the estimates of this model:

1. Raw data
2. Data preparation: filters from the raw records to the estimation sample (records kept and dropped), rebuild check
3. Estimation data and choice set
4. Descriptive statistics and figures; 4.1 every explanatory variable of the model in the data
5. Model specification (the folder's `model/estimate_car_ownership.py`, unchanged)
6. Correlation of the explanatory variables (heatmaps, VIF)
7. Log-likelihood at the article coefficients
8. Biogeme estimation
9. Results: article format, fit, comparison with the article, correlation of the estimates
10. Hold-out sample

11. Glossary

## Model at a glance

**1. Who decides.** Maryland households (not group quarters). Data: ACS PUMS 2014-2018 (5-year), Maryland. Estimation sample: 91,348 households.

**2. What is chosen (dependent variable).** Number of cars. Alternatives: 0, 1, 2, 3 or more cars (all available; no car is the reference with utility 0).

**3. Explanatory variables** (signs of the article estimates, `model/reference_estimates.csv`):

| Group | Variable | What it measures | 1 car | 2 cars | 3+ cars |
|---|---|---|---|---|---|
| Constant | Constant | baseline preference against no car | + | − | − |
| Household | Workers | number of workers | + | + | + |
| Household | Persons aged 16+: 2 | against one person aged 16+ | − | + | + |
| Household | Persons aged 16+: 3 | against one | − | + | + |
| Household | Persons aged 16+: 4 or more | against one | − | + | + |
| Household | Children under 16 | number of children | − | − | − |
| Household | Single-person household | one person | − | 0 (fixed) | 0 (fixed) |
| Household | Age of the oldest member | (age - 45) / 10 | − | − | 0 (fixed) |
| Household | Age of the oldest member, squared | ((age - 45) / 10)^2 | − | − | − |
| Income | ln income, under $25k | piece of ln income | + | + | + |
| Income | ln income, $25-50k | piece of ln income | + | + | + |
| Income | ln income, $50-100k | piece of ln income | + | + | + |
| Income | ln income, above $100k | piece of ln income | − | + | + |
| Dwelling | Single-family detached | against attached and other | + | + | + |
| Dwelling | Multifamily | against attached and other | − | − | − |
| Dwelling | Bedrooms | number of bedrooms | + | + | + |
| Location | ln dwelling density of the PUMA | per km2 | 0 (fixed) | − | − |
| Location | Accessibility gain from a car | non-work logsum with minus without a car | + | + | + |
| Location | ln income x ln density | centred interaction | + | + | − |
| Location | Baltimore City | home in Baltimore City | − | − | − |

**4. How household traits enter.** Household traits enter directly, with one coefficient per number of cars; no car is the reference with utility 0, so each coefficient is a difference against owning no car.

**5. How to read a coefficient.** A positive coefficient makes the alternative more likely, a negative one less likely. exp(coefficient) is the factor by which the odds change for one unit more of the variable.
 Example: Workers = +0.332, odds factor exp(0.332) = 1.394 per unit.

In [ ]:
# switches (see How to run) and set-up: imports, folders, helper modules
RUN_DATA_PREP  = True    # rebuild the estimation sample and tables into output/rebuild_check and compare with the shipped tables (section 2)
RUN_CHECK      = True    # log-likelihood at the article coefficients
RUN_ESTIMATION = False   # Biogeme estimation, written to output/ (False: results are read from output/ if there are any)
RUN_HOLDOUT    = True    # log-likelihood of the hold-out sample (at your estimates if there are any, else at the article coefficients)

import os, sys, json, glob, inspect, importlib, time, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import biogeme.biogeme as bio
from biogeme.parameters import Parameters
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt; import nb_reader as rd; import figs as fg
fg = importlib.reload(fg)
from IPython.display import Markdown
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "car_ownership"; MODEL_FOLDER = "04_car_ownership"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", os.path.basename(FOLDER))

## 1. Raw data

*Where the records come from. Look for: the files and the record counts.*

In [ ]:
# Raw data
SH = os.path.abspath(os.path.join(FOLDER, "..", "00_shared_data")); CODE = os.path.join(FOLDER, "input", "code")
RAW = os.path.join(FOLDER, "input", "data", "raw"); REBUILD = os.path.join(OUT, "rebuild_check")
ZS = pd.read_csv(os.path.join(SH, "zone_system", "zoneSystem.csv"))[["ZoneId", "COUNTYFIPS", "JUR_NAME", "PUMA_10"]].rename(columns={"ZoneId": "zone"})
ZS["area_group"] = nt.area_group_of_county(ZS.COUNTYFIPS)

def rebuild_tables(python=sys.executable, pre=""):
    """build_estimation_data.main() of input/code with its output folder set to a scratch folder; returns the comparison with the
    shipped estimation and hold-out tables."""
    out = os.path.join(REBUILD, "data"); [os.makedirs(os.path.join(out, h), exist_ok=True) for h in ("estimation", "holdout")]
    nt.run(f'{python} -c "import build_estimation_data as B; {pre}B.OUT = {out!r}; B.main()"', cwd=CODE)
    return pd.concat([nt.compare_trees(os.path.join(out, h), os.path.join(FOLDER, "input", "data", h)).assign(half=h) for h in ("estimation", "holdout")],
                     ignore_index=True)

### 1.1 ACS PUMS 2014-2018, Maryland

The car ownership model uses the Maryland households of the ACS PUMS 2014-2018 (`csv_hmd.zip`, `csv_pmd.zip`). A record is a
household with its number of vehicles (0, 1, 2, 3+), persons, workers, income, dwelling and the accessibility of its PUMA.

In [ ]:
# 1.1 ACS PUMS 2014-2018, Maryland
import zipfile
def pums(kind, cols):
    with zipfile.ZipFile(os.path.join(SH, "acs_pums_2014_2018_5yr", f"csv_{kind}md.zip")) as zf:
        n = [x for x in zf.namelist() if x.endswith(".csv")][0]
        return pd.read_csv(zf.open(n), usecols=cols, low_memory=False, dtype={"SERIALNO": str})
H = pums("h", ["SERIALNO", "PUMA", "NP", "VEH", "HINCP", "WGTP", "TYPE", "ADJINC", "BLD", "BDSP"])
print(f"Maryland PUMS household records: {len(H):,}")
H.head()

## 2. Data preparation

*From the raw records to the estimation sample. Look for: the records kept and dropped at each filter. With RUN_DATA_PREP the rebuilt tables must equal the shipped ones.*

### 2.1 From PUMS records to households

`input/code/s01_data.py` keeps households (TYPE 1) with persons, a known vehicle count and a positive weight, adds persons,
workers, income (2016 $), dwelling type and the PUMA accessibility, drops records with a missing value, and splits by a hash
of the serial number (`md5 mod 5 = 0` = hold-out, 20 %). Mirrored below.

In [ ]:
# Data preparation
HH = pd.read_parquet(os.path.join(RAW, "hh.parquet"))
f = nt.Funnel("Maryland PUMS household records", len(H))
x = H[H.TYPE == 1]; f.add("households (TYPE 1)", len(x), "group quarters")
x = x[x.NP > 0]; f.add("with persons", len(x), "vacant units")
x = x[x.VEH.notna() & (x.WGTP > 0)]; f.add("vehicles known, positive weight", len(x), "")
f.add("no missing value (hh.parquet)", len(HH), "missing accessibility or density")
f.add("estimation half (80 %)", int((HH.half == "estimation").sum()), "hold-out half (20 %)")
display(f.table()); nt.funnel_plot(f, "households")

### 2.2 Rebuild check

`s01_data.main()` and `build_estimation_data.main()` run with their output folders set to a scratch folder; the household file and the estimation and hold-out tables are compared with the shipped ones.

In [ ]:
# 2.2 Rebuild check
if RUN_DATA_PREP:
    t0 = time.time(); R1 = os.path.join(REBUILD, "raw"); os.makedirs(R1, exist_ok=True)
    nt.run(f'{sys.executable} -c "import s01_data as S; S.RAWD = {R1!r}; S.main()"', cwd=CODE, quiet=True)
    res_h = nt.compare_frames(pd.read_parquet(os.path.join(R1, "hh.parquet")), HH, name="hh.parquet")
    res = rebuild_tables(pre=f"B.RAWD = {R1!r}; ")
    res = pd.concat([res_h, res], ignore_index=True); print(f"rebuild: {(time.time() - t0) / 60:.1f} min"); display(res)
    nt.rebuild_verdict(res, glob.glob(os.path.join(FOLDER, "model", "estimate_*.py"))[0])
else:
    print("skipped (RUN_DATA_PREP = False)")

### Data sources

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

**ACS PUMS 2014-2018 5-year, household and person files** — U.S. Census Bureau
- Where: https://www2.census.gov/programs-surveys/acs/data/pums/2018/5-Year/
- How: Download csv_h<st>.zip and csv_p<st>.zip for st = dc, de, md, pa, va, wv from the 2018 5-Year folder (CSV, not SAS). Variables used: SERIALNO, ST, PUMA, MIGPUMA, MIGSP, POWPUMA, POWSP, MIG, MV, RELP, AGEP, ESR, JWMNP, SCHG, NP, TYPE, BDSP, BLD, TEN, GRNTP, SMOCP, VALP, HINCP, VEH, YBL, WGTP; ADJINC (income adjustment) and ADJHSG (housing-cost adjustment) are columns of the same files.
- Put in: 00_shared_data/raw/acs_pums_2014_2018_5yr/
- Files: `csv_hdc.zip`, `csv_hde.zip`, `csv_hmd.zip`, `csv_hpa.zip`, `csv_hva.zip`, `csv_hwv.zip`, `csv_pdc.zip`, `csv_pde.zip`, `csv_pmd.zip`, `csv_ppa.zip`, `csv_pva.zip`, `csv_pwv.zip`

**MSTM zone system, zone polygons and peak car skim (Maryland Statewide Transportation Model)** — Maryland Department of Transportation State Highway Administration (MDOT SHA), provided to the project
- Where: https://roads.maryland.gov
- How: Not a public download: the 1,588-zone MSTM system (zoneSystem.csv), zone polygons (SMZ_RMZ_02152011inMSTM_EPSG26918.shp) and the AM-peak auto travel-time skim were provided by MDOT. Request them from MDOT SHA (MSTM / travel forecasting).
- Put in: 00_shared_data/raw/mstm/
- Files: `zoneSystem.csv`, `SMZ_RMZ_02152011inMSTM_EPSG26918.shp/.shx/.dbf/.prj`, `auto_peak_time.csv.gz`

**CPI-U, U.S. city average, all items (annual averages 2014-2023)** — U.S. Bureau of Labor Statistics
- Where: https://data.bls.gov/timeseries/CUUR0000SA0
- How: Series CUUR0000SA0, annual averages; the values are typed in rc.py (CPI dict) and used to express incomes in 2016 dollars.
- Put in: 00_shared_data/raw/cpi/

**Regional Travel Survey 2017/18 (household, person, trip records)** — Baltimore Metropolitan Council (BMC) and National Capital Region Transportation Planning Board (TPB/MWCOG)
- Where: https://www.baltometro.org
- How: Restricted: request the 2017/2018 Regional Travel Survey data from BMC / TPB. File used: RTS_raw_flat_data.csv (flat person-level file with household fields: household_id, person_id, home_state_county_fips, home_tract_fips, home_type, hhsize, numvehicle, numdrivers, hh_income_detailed, wthhfin, wtperfin, age, age_group, employment_status, j1_workplace_loc, work_state_county_fips, work_tract_fips, license, school locations and BMC/TPB TAZ codes). Also rts_trips_clean.csv (cleaned trips) for the synthetic-population workplace step.
- Put in: 00_shared_data/raw/rts_2017_18/
- Files: `RTS_raw_flat_data.csv`, `rts_trips_clean.csv`

**LEHD LODES 8, Workplace Area Characteristics 2016 and geography crosswalk, Maryland** — U.S. Census Bureau, LEHD
- Where: https://lehd.ces.census.gov/data/lodes/LODES8/md/wac/md_wac_S000_JT00_2016.csv.gz
- How: Download md_wac_S000_JT00_2016.csv.gz and https://lehd.ces.census.gov/data/lodes/LODES8/md/md_xwalk.csv.gz (other states: .../LODES8/<st>/wac/<st>_wac_S000_JT00_<year>.csv.gz). Workplace zone attributes (high-earning share CE03/C000) use the 2017 WAC files of md, dc, va, pa, de, wv: https://lehd.ces.census.gov/data/lodes/LODES8/<st>/wac/<st>_wac_S000_JT00_2017.csv.gz (project: data/raw/lodes_wac/ and data/raw/lodes_wac_2017_extra/).
- Put in: 00_shared_data/raw/lodes_wac/
- Files: `md_wac_S000_JT00_2016.csv.gz`, `md_xwalk.csv.gz`, `<st>_wac_S000_JT00_2017.csv.gz (st = md, dc, va, pa, de, wv)`

**BLS JOLTS job-openings rate, Maryland** — U.S. Bureau of Labor Statistics
- Where: https://api.bls.gov/publicAPI/v1/timeseries/data/JTS000000240000000JOR
- How: BLS public API v1, series JTS000000240000000JOR (Maryland) and JTS000000000000000JOR (U.S.), annual mean of monthly SA values.
- Put in: 00_shared_data/raw/jolts/
- Files: `jolts_bls_api.json`

**Census 2010 Redistricting (PL 94-171) Maryland file and TIGER 2010 blocks and tracts** — U.S. Census Bureau
- Where: https://www2.census.gov/census_2010/01-Redistricting_File--PL_94-171/Maryland/
- How: md2010.pl.zip (geo file and segment 2, housing units H1) and TIGER/Line 2010 tl_2010_24_tabblock10.zip; tracts tl_2010_<st>_tract10.zip for st = 10, 11, 24, 42, 51, 54 (https://www2.census.gov/geo/tiger/TIGER2010/TRACT/2010/).
- Put in: 00_shared_data/raw/census2010/
- Files: `md2010.pl.zip`, `tl_2010_24_tabblock10.zip`, `tl_2010_<st>_tract10.zip`

## 3. Estimation data and choice set

*The table the model reads. Look for: the columns used and what they mean (data dictionary).*

Alternatives: 0, 1, 2, 3+ cars (ordered as an MNL with alternative-specific coefficients), all available.

The table exactly as used in the estimation, and its data dictionary.

In [ ]:
# Estimation data and choice set
est = os.path.join(FOLDER, "input", "data", "estimation")
display(nt.files_table(est))
dd = pd.read_csv(os.path.join(est, "data_dictionary.csv"))
dd

In [ ]:
# Estimation data and choice set
import estimate_car_ownership as M
df = pd.read_csv(M.DATA)
print(f"{df.shape[0]:,} rows (decision makers), {df.shape[1]:,} columns")
df.head()

## 4. Descriptive statistics

*What the sample looks like. Look for: plausible shares and maps, and the chosen alternatives against the available ones.*

In [ ]:
# Observed cars per household, unweighted and with the PUMS household weight (the estimation is unweighted)
HO = pd.read_csv(os.path.join(FOLDER, "input", "data", "holdout", "car_ownership_holdout.csv"))
W_ = HH[HH.half == "estimation"].set_index("serial").reindex(df.serial.astype(str).values)
assert W_.w.notna().all()
sh = nt.shares_table(df.cars.map({0: "0", 1: "1", 2: "2", 3: "3+"}), W_.w.values)
display(sh)
nt.grouped_bars(sh[["share %", "weighted share %"]].rename(columns={"share %": "unweighted", "weighted share %": "household weight (WGTP)"}), "cars of the household", "share of households (%)", colors=[nt.OBS, nt.LIGHT])
plt.tight_layout(); plt.show()

Observed cars per household, unweighted and with the PUMS household weight (the estimation is unweighted).

In [ ]:
# Households of the estimation half: persons, workers, children, income, age, dwelling, density and the accessib
fig, ax = nt.panels(8, ncol=4, w=3.4, h=2.8)
nt.bar((100 * W_.hhsize.clip(upper=6).value_counts(normalize=True)).sort_index(), "persons (6 = 6+)", "share (%)", ax=ax[0])
nt.bar((100 * df.workers.clip(upper=4).value_counts(normalize=True)).sort_index(), "workers (4 = 4+)", "share (%)", ax=ax[1])
nt.bar((100 * df.children.clip(upper=4).value_counts(normalize=True)).sort_index(), "children under 16 (4 = 4+)", "share (%)", ax=ax[2])
nt.hist(W_.income_k.clip(upper=400), "household income (2016 k$)", "households", bins=50, ax=ax[3])
nt.hist(df.age10 * 10 + 45, "age of the oldest member (years)", "households", bins=range(15, 96, 5), ax=ax[4])
dw = pd.Series(np.select([df.sfd == 1, df.multifamily == 1], ["single-family detached", "multi-family"], "other")).value_counts()
nt.bar(100 * dw / dw.sum(), "dwelling", "share (%)", ax=ax[5], rot=20)
nt.hist(df.ln_density, "ln density of the PUMA", "households", bins=40, ax=ax[6])
nt.hist(df.acc_gain, "accessibility gain of a car", "households", bins=40, ax=ax[7])
plt.tight_layout(); plt.show()
print("share without a car (%) by workers:", (100 * df.groupby(df.workers.clip(upper=3)).cars.apply(lambda c: (c == 0).mean())).round(1).to_dict())

Households of the estimation half: persons, workers, children, income, age, dwelling, density and the accessibility gain of owning a car.

In [ ]:
# Share of households without a car by PUMA (Maryland), and the car shares of the two halves
r = df.groupby("puma7").cars.apply(lambda c: 100 * (c == 0).mean())
z = ZS.set_index("zone").PUMA_10.map(r).dropna()
fig, ax = plt.subplots(figsize=(7.5, 6)); nt.zone_map(z, "households without a car (%), by PUMA", ax=ax, zoom="md"); plt.tight_layout(); plt.show()
nt.half_comparison(df.cars, HO.cars, "cars")

Share of households without a car by PUMA (Maryland), and the car shares of the two halves.

In [ ]:
# Descriptive statistics
pass

### 4.1 Variables in the data

Each explanatory variable of the final specification as it enters the utility, interactions included (read from the model's own utility functions: the part of the utility that each coefficient multiplies; coefficients that share a label, e.g. the copies of a term for several purposes, are shown in one panel). Here: each variable by the number of cars the household owns (the chosen alternative); unweighted.

In [ ]:
# every explanatory variable of the specification by chosen number of cars
import varplots as vp
_db, _ll = M.model(); U_, AV_, CH_ = vp.find_choice(_ll)
ALT_NAMES = {0: "No car", 1: "1 car", 2: "2 cars", 3: "3+ cars"}
VP = vp.by_choice(U_, AV_, CH_, df, vp.silo_labels(MODEL_FOLDER), ALT_NAMES)
display(Markdown(vp.choice_counts_line(vp.choices(CH_, df), ALT_NAMES, "households")))
vp.by_choice_figures(VP, list(ALT_NAMES.values()), unit="households")

How to read it: one bar per number of cars: the mean of the variable (or the share with value 1) among the households that own that many cars; the grey range is the 25th-75th percentile, the dashed line the mean of all households. A variable enters the utilities of 1, 2 and 3+ cars (no car is the reference) with one coefficient each.

## 5. Model specification

*The utility functions, read from the model script; nothing is re-specified here.*

The specification is the folder's script `model/estimate_car_ownership.py`, imported as it is. The start values are the article estimates.

In [ ]:
# Model specification
print(inspect.getsource(M.model))

In [ ]:
# Model specification
t0 = time.time()
database, ll = M.model()
used = nt.model_variables(ll)
print(f"database: {database.num_rows():,} rows; {len(used)} data columns used by the model; built in {time.time()-t0:.1f} s")

## 6. Correlation of the explanatory variables

*Correlation of the explanatory variables. Look for: |r| of 0.7 or more and VIF of 10 or more; these make single coefficients less precise.*

Correlations of the variables that enter the utility, computed over the alternatives available to each decision maker
(alternative-specific variables) and over decision makers (person and household variables). |r| ≥ 0.7 and VIF ≥ 10 are
listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
# Correlation of the explanatory variables
Xu = nt.explanatory(df[[c for c in used if c in df.columns and c != "cars"]])
Cm = nt.corr_heatmap(Xu, "Household variables of the car ownership model")
display(nt.high_pairs(Cm)); display(nt.vif(Xu))

## 7. Log-likelihood at the article coefficients

*Log-likelihood at the article coefficients, without estimation. Look for: the two numbers should be equal (to rounding).*

No estimation: the log-likelihood at the start values (the article estimates). Target: article -78613.6,
Biogeme -78613.6202 (N = 91348, K = 56).

In [ ]:
# Log-likelihood at the article coefficients
if RUN_CHECK:
    p = Parameters(); p.set_value("generate_html", False); p.set_value("generate_yaml", False)
    here = os.getcwd(); os.chdir(__import__("tempfile").mkdtemp())
    t0 = time.time()
    b = bio.BIOGEME(database, ll, parameters=p); b.model_name = NAME + "_check"
    ll0 = b.calculate_init_likelihood()
    os.chdir(here)
    print(f"N {database.num_rows():,}   LL at the article coefficients: {ll0:,.4f}   (target -78613.6202)   {time.time()-t0:.0f} s")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

*Biogeme estimation, only with RUN_ESTIMATION. Look for: convergence and the run time.*

The same call as `model/estimate_car_ownership.py`: Biogeme with its defaults (read from `output/biogeme.toml`), start values = article estimates.
Report written to `output/car_ownership.html`, `.yaml`, and `output/car_ownership_estimates.csv`.

In [ ]:
# Biogeme estimation
if RUN_ESTIMATION:
    here = os.getcwd(); os.chdir(OUT)
    t0 = time.time()
    b = bio.BIOGEME(database, ll); b.model_name = NAME
    results = b.estimate()
    os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s"); print(results.short_summary())
    nt.estimates_table(results)[["parameter", "coefficient", "robust_se", "t", "p"]].rename(
        columns={"coefficient": "Value", "robust_se": "Robust std err.", "t": "Robust t-stat.", "p": "Robust p-value", "parameter": "Name"}
    ).to_csv(os.path.join(OUT, f"{NAME}_estimates.csv"), index=False)
else:
    ref_time = os.path.getmtime(os.path.join(FOLDER, "model", "reference_estimates.csv"))
    y = [f for f in sorted(glob.glob(os.path.join(OUT, f"{NAME}*.yaml")), key=os.path.getmtime) if os.path.getmtime(f) > ref_time and "_check" not in f]
    results = EstimationResults.from_yaml_file(y[-1]) if y else None
    print(f"RUN_ESTIMATION = False: read {os.path.basename(y[-1])} (newer than model/reference_estimates.csv)" if y else "RUN_ESTIMATION = False and no current result in output/: sections 9-10 are skipped")

## 9. Results

*The estimates and what they mean. Look for: coefficients equal to the article within rounding, and the readings below.*

### Estimates in the article format
Coefficient (stars: *** p < 0.01, ** p < 0.05, * p < 0.10, as in the article), robust s.e., p.

In [ ]:
# Results
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### What the coefficients say

One plain line per term, from your estimates (from the article estimates if there are none). Income interactions are given at $25k, $50k and $100k; odds change = exp(coefficient) per unit.

In [ ]:
# plain readings of the coefficients and the coefficient plot (95 % robust intervals, grouped by block)
est_v = est_t if results is not None else rd.reference(MODEL_FOLDER)[0]
display(rd.readings(MODEL_FOLDER, est_v).style.hide(axis="index").set_properties(**{"text-align": "left"}))
fg.coef_figure(fg.coef_rows(MODEL_FOLDER, est_v)); plt.show()

How to read it: a point right of zero makes the alternative more likely; an interval that crosses zero is not significant.

### Goodness of fit

In [ ]:
# Goodness of fit
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
# Comparison with the article (model/reference_estimates.csv)
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "N": database.num_rows() if "database" in dir() else None,
                                                   "K": results.number_of_free_parameters})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
# Correlation of the estimates
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

*Fit on the hold-out records, which the estimation did not use. Look for: a log-likelihood per record close to that of the estimation sample, and the maps.*

In [ ]:
# Hold-out sample
if RUN_HOLDOUT:
    import tempfile
    est_data = M.DATA
    M.DATA = os.path.join(FOLDER, "input", "data", "holdout", 'car_ownership_holdout.csv')
    at = "your estimates" if results is not None else "the article coefficients"
    try:
        hdb, hll = M.model(start=results.get_beta_values() if results is not None else None)
    finally:
        M.DATA = est_data
    p = Parameters(); p.set_value("generate_html", False); p.set_value("generate_yaml", False)
    here = os.getcwd(); os.chdir(tempfile.mkdtemp())
    b = bio.BIOGEME(hdb, hll, parameters=p); b.model_name = NAME + "_holdout"
    llh = b.calculate_init_likelihood(); os.chdir(here)
    print(f"hold-out: N {hdb.num_rows():,}   LL at {at} {llh:,.2f}   mean LL per observation {llh/hdb.num_rows():.4f}")
    if results is not None:
        print(f"estimation sample: mean LL per observation {results.final_loglikelihood/database.num_rows():.4f}")
else:
    print("skipped (RUN_HOLDOUT = False)")

### Hold-out shares by number of cars and by household

Observed and predicted shares of the hold-out households (predicted = mean of the probabilities).

In [ ]:
# predicted against observed car shares on the hold-out, overall, by workers and by dwelling (article coefficients if no estimates)
est_data = M.DATA; M.DATA = os.path.join(FOLDER, "input", "data", "holdout", "car_ownership_holdout.csv")
try:
    hdb, hll = M.model(); hdf = pd.read_csv(M.DATA)
finally:
    M.DATA = est_data
ref_ = rd.reference(MODEL_FOLDER)[0]
betas = results.get_beta_values() if results is not None else dict(zip(ref_.parameter, ref_.coefficient))
P = nt.predicted_probabilities(hdb, hll, betas)
fg.obs_model_figure(fg.cars_holdout_panels(hdf, P)); plt.show()

How to read it: dark grey observed, orange model; (b) and (c) test the share without a car within groups of households.

## Glossary

- **PUMA**: Public Use Microdata Area: a Census area of about 100,000 people, the smallest area of the ACS microdata.
- **reference alternative**: The alternative whose utility is fixed at 0; the coefficients of the others are differences against it.
- **log-likelihood**: Sum over decision makers of ln(probability of the observed choice). Closer to 0 is better.
- **BIC**: Bayesian information criterion: -2 log-likelihood + K ln N. Lower is better; it penalises extra parameters.
- **hold-out**: The 20 % (residential, cars) or half (survey models) of the records not used for estimation. It was also used in the specification search, so its fit is not a fully independent test.
- **robust s.e.**: Standard error from the sandwich formula, valid when the model is not exactly right.
- **VIF**: Variance inflation factor: how much the correlation with other variables inflates a coefficient's variance; 10 or more is high.